In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
# !wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!cp /content/drive/MyDrive/MMDS-data/spark-3.1.1-bin-hadoop3.2.tgz .
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!cp /content/drive/MyDrive/MMDS-data/spark-3.1.1-bin-hadoop3.2.gz .
!tar xf spark-3.1.1-bin-hadoop3.2.gz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .appName("task3endterm") \
        .getOrCreate()

sc = spark.sparkContext
ratings2k = "/content/drive/MyDrive/MMDS-data/ratings2k.csv"

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.mllib.linalg import Vectors
from pyspark.mllib.linalg.distributed import RowMatrix

df = spark.read.csv(ratings2k, header=True, inferSchema=True)
df.show(10)

+-----+----+----+------+
|index|user|item|rating|
+-----+----+----+------+
|    0|  73|  52|   4.0|
|    1|  36| 239|   3.0|
|    2|  72|  26|   1.0|
|    3|  59| 430|   2.5|
|    4|  72| 284|   3.0|
|    5|  36| 277|   3.0|
|    6|  72| 426|   4.0|
|    7|  18| 163|   3.0|
|    8|  67|  93|   4.0|
|    9|  59|  22|   3.5|
+-----+----+----+------+
only showing top 10 rows



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.linalg import Vectors
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType
from pyspark.ml.linalg import DenseVector

class CollaborativeFiltering:
    def __init__(self, N, ratings_df, user_id, top_n):
        self.N = N
        self.ratings_df = ratings_df
        self.user_id = user_id
        self.top_n = top_n

    def _calculate_similarity(self, user_id):
        user_ratings = self.ratings_df.groupBy("user").agg(F.collect_list(F.struct("item", "rating")).alias("ratings"))

        target_user_ratings = user_ratings.filter(F.col("user") == user_id).select("ratings").collect()
        if not target_user_ratings:
            return None

        target_user_ratings = target_user_ratings[0]["ratings"]

        similarities = []
        for row in user_ratings.collect():
            other_user_id = row["user"]
            if other_user_id == user_id:
                continue

            other_user_ratings = row["ratings"]
            common_items = set([x["item"] for x in target_user_ratings]) & set([x["item"] for x in other_user_ratings])

            if common_items:
                target_ratings = [x["rating"] for x in target_user_ratings if x["item"] in common_items]
                other_ratings = [x["rating"] for x in other_user_ratings if x["item"] in common_items]

                similarity = self._calculate_cosine_similarity(target_ratings, other_ratings)
                similarities.append((other_user_id, similarity))

        top_similar_users = sorted(similarities, key=lambda x: x[1], reverse=True)[:self.N]
        return top_similar_users

    def _calculate_cosine_similarity(self, vec1, vec2):
        vec1 = DenseVector(vec1)
        vec2 = DenseVector(vec2)
        dot_product = float(vec1.dot(vec2))
        norm1 = float(vec1.norm(2))
        norm2 = float(vec2.norm(2))
        if norm1 > 0 and norm2 > 0:
            return dot_product / ((norm1 * norm2) + 1e-10)
        else:
            return 0.0

    def predict(self, user_id):
        similar_users = self._calculate_similarity(user_id)
        if not similar_users:
            return None

        item_ratings = {}
        for other_user_id, similarity in similar_users:
            other_user_ratings = self.ratings_df.filter(F.col("user") == other_user_id).collect()
            for row in other_user_ratings:
                item = row["item"]
                rating = row["rating"]
                if item not in item_ratings:
                    item_ratings[item] = {"weighted_sum": 0, "weight_sum": 0}

                item_ratings[item]["weighted_sum"] += rating * similarity
                item_ratings[item]["weight_sum"] += abs(similarity)

        predictions = []
        for item, values in item_ratings.items():
            if values["weight_sum"] > 0:
                predicted_score = values["weighted_sum"] / values["weight_sum"]
                predictions.append((item, round(predicted_score, 3)))


        recommendations_df = spark.createDataFrame(predictions, ["item", "predicted_score"])
        top_recommendations = recommendations_df.orderBy(F.desc("predicted_score")).limit(self.top_n)
        return top_recommendations

    def run(self):
        recommendation = self.predict(self.user_id)
        recommendation.write.csv("output/recommendations_user_" + str(self.user_id), header=True, mode="overwrite")
        return recommendation

recommender = CollaborativeFiltering(N=20, ratings_df=df, user_id=1, top_n=100)
top_recommendations = recommender.run()
top_recommendations.show()

+----+---------------+
|item|predicted_score|
+----+---------------+
| 117|            5.0|
| 259|            5.0|
| 431|            5.0|
| 164|            5.0|
|  99|            5.0|
| 391|            5.0|
| 183|            5.0|
| 110|            5.0|
| 465|            5.0|
| 267|            5.0|
| 107|            5.0|
| 135|            5.0|
|  84|            5.0|
| 388|            5.0|
|  54|            5.0|
|  53|            5.0|
| 355|            5.0|
| 354|            5.0|
| 163|          4.868|
| 413|           4.75|
+----+---------------+
only showing top 20 rows

